<div align="center">
  <span style="font-size:50px; color:maroon;"><b>
      Coeficiente barometrico
  </b>
  </span>
</div>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import os 
import glob 

# Definimos la carpeta con la que queremos trabajar 
NOMBRE_CARPETA = 'MonthlyDataCompletalim' 

# Ruta absoluta a la carpeta
RUTA_CARPETA = os.path.join(os.getcwd(), NOMBRE_CARPETA)
# Busca todos los archivos '.txt'
patron_busqueda = os.path.join(RUTA_CARPETA, '*.txt')
archivos_mensuales = glob.glob(patron_busqueda)
archivos_mensuales.sort() 

# Diccionario de nombres de meses 
nombres_meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio", 
    7: "Julio", 8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

n_archivos = len(archivos_mensuales)
# Define las dimensiones de la cuadrícula
n_rows = int(np.ceil(n_archivos / 3)) 
n_cols = 3 

fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 6 * n_rows))
axes = axes.flatten() 

Betas = 0
meses_contados = 0
plot_index = 0 


# Procesamos y graficamos cada archivo 
for ruta_completa in archivos_mensuales:
    # Línea corregida manualmente para evitar el error U+00A0
    nombre_archivo = os.path.basename(ruta_completa)
    print(f"\n Los datos del archivo {nombre_archivo} son")
    
    # Lectura de Datos 
    df_messelecc = pd.read_csv(ruta_completa, sep='\t')
    
    mes_num = int(nombre_archivo[:2]) 
    anio_str = nombre_archivo[2:4] 
    nombre_mes = nombres_meses.get(mes_num, f"Mes Desconocido ({mes_num})")
    titulo_subplot = f'{nombre_mes} - 20{anio_str}'
    
    # Iniciamos con nuestros calculos
    df_messelecc = df_messelecc.dropna(subset=['Data', 'Pressure(mb)']).copy()
    
    # Cálculo de Promedios y Deltas
    PromedioICN = df_messelecc['Data'].mean()
    PromedioPress = df_messelecc['Pressure(mb)'].mean()

    df_messelecc['DeltaP'] = df_messelecc['Pressure(mb)'] - PromedioPress
    df_messelecc['DeltaN'] = np.log(df_messelecc['Data'] / PromedioICN) * 100

    # Ajuste lineal (Regresión)
    Beta, b, r_value, p_value, std_err = stats.linregress(df_messelecc['DeltaP'], df_messelecc['DeltaN'])
    
    Error = std_err
    Betas += Beta
    meses_contados += 1
    
    # Cálculo de Regresión y Estadísticas
    Regresion = Beta * df_messelecc['DeltaP'] + b
    R2 = r_value**2
    Pearson, _ = stats.pearsonr(df_messelecc['DeltaP'], df_messelecc['DeltaN']) 
    
    # Formato de texto para la gráfica
    equation = fr'y=({Beta:.6f} $\pm$ {Error:.4f}) x + ({b:.1e})'
    R2_str = f'$R^2$={R2:.3f}'
    r_str = f'pearson r={Pearson:.2f}'

    if plot_index < len(axes):
        ax = axes[plot_index]
        
        ax.scatter(df_messelecc['DeltaP'], df_messelecc['DeltaN'], marker='o', color='indianred', edgecolor='brown', linewidth=0.1)
        ax.plot(df_messelecc['DeltaP'], Regresion, color='black')
        ax.grid(True, linestyle=':', alpha=0.6)
        
        # Etiquetas usando notación LaTeX
        ax.set_xlabel(r'$\Delta P \ (\text{mb})$', fontsize=10, fontfamily='serif')
        ax.set_ylabel(r'$log \left(\dfrac{N}{N_0}\right) \ (\%)$', fontsize=10, fontfamily='serif') 

        ax.set_title(titulo_subplot, fontsize=12, fontfamily='serif')
        
        ax.text(0.02, 0.97, R2_str, transform=ax.transAxes, fontsize=9, fontfamily='serif', verticalalignment='top')
        plot_index += 1

    print(equation)
    print(R2_str)
    print(r_str)
    print(f"Beta para {nombre_mes}: {Beta:.6f}")

for i in range(plot_index, len(axes)):
    fig.delaxes(axes[i])
    
plt.tight_layout()

# Guardar la figura
plt.savefig("SubplotsCalulosBarometricos25_Completo", dpi=300, bbox_inches='tight')

plt.show()

print("-" * 50)
if meses_contados > 0:
    beta_promedio = Betas / meses_contados
    print(f"El promedio de Beta de los {meses_contados} meses es: {beta_promedio:.6f}")

ValueError: Number of rows must be a positive integer, not 0

<Figure size 2400x0 with 0 Axes>